# V1

In [ ]:
from urllib.parse import unquote
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set, Tuple

from rdflib import Graph, Literal, URIRef
from rdflib.namespace import OWL, RDF, RDFS, XSD


# ----------------------------
# Data model
# ----------------------------

@dataclass
class ClassDef:
    iri: str                                                    # Full IRI: eg. http://dblp.org/rdf/schema#Person
    labels: List[str] = field(default_factory=list)             # For human readable names
    comments: List[str] = field(default_factory=list)           # rdfs:comment descriptions

    super_classes: Set[str] = field(default_factory=set)        # rdfs:subClassOf
    equivalent_classes: Set[str] = field(default_factory=set)   # owl:equivalentClass
    disjoint_with: Set[str] = field(default_factory=set)        # owl:disjointWith


@dataclass
class PropDef:
    iri: str
    labels: List[str] = field(default_factory=list)
    comments: List[str] = field(default_factory=list)

    domains: Set[str] = field(default_factory=set)              # What classes can use this property (rdfs:domain)
    literal_datatypes: Set[str] = field(default_factory=set)    # rdfs:range in XSD namespace (range is literal)
    range_classes: Set[str] = field(default_factory=set)        # rdfs:range for non XSD IRIs (range is IRI)

    super_properties: Set[str] = field(default_factory=set)     # rdfs:subPropertyOf
    equivalent_properties: Set[str] = field(default_factory=set)# owl:equivalentProperty
    inverses: Set[str] = field(default_factory=set)             # owl:inverseOf (Inverse relations eg. authorOf <-> hasAuthor)
    property_types: Set[str] = field(default_factory=set)       # rdf:type values like owl:ObjectProperty, owl:FunctionalProperty, etc


@dataclass
class SchemaIndex:
    namespaces: Dict[str, str] = field(default_factory=dict)    # prefix -> namespace (eg. "dblp" -> "http://dblp.org/rdf/schema#")
    classes: Dict[str, ClassDef] = field(default_factory=dict)  # iri -> class def
    props: Dict[str, PropDef] = field(default_factory=dict)     # iri -> property def


# ----------------------------
# CURIE helpers
# ----------------------------

def _ensure_default_prefixes(namespaces: Dict[str, str]) -> Dict[str, str]:
    defaults = {"rdf": str(RDF), "rdfs": str(RDFS), "owl": str(OWL), "xsd": str(XSD)}
    out = dict(namespaces)
    for k, v in defaults.items():
        out.setdefault(k, v)
    return out


def make_curie(iri: str, namespaces: Dict[str, str]) -> str:
    iri_u = unquote(iri)
    best: Optional[Tuple[str, str]] = None
    best_len = -1
    for pfx, ns in namespaces.items():
        ns_u = unquote(ns)
        if iri_u.startswith(ns_u) and len(ns_u) > best_len:
            best = (pfx, ns_u)
            best_len = len(ns_u)
    if best is None:
        return iri
    pfx, ns_u = best
    return f"{pfx}:{iri_u[best_len:]}"


# ----------------------------
# Load schema
# ----------------------------

def load_schema(path: str, base_iri: Optional[str] = None) -> SchemaIndex:
    g = Graph()
    if base_iri is None:
        g.parse(path)
    else:
        g.parse(path, publicID=base_iri)

    ns: Dict[str, str] = {pfx: str(uri) for pfx, uri in g.namespaces()}
    ns = _ensure_default_prefixes(ns)
    idx = SchemaIndex(namespaces=ns)

    # -------- classes --------
    class_nodes: Set[URIRef] = set()
    for s in g.subjects(RDF.type, OWL.Class):
        if isinstance(s, URIRef):
            class_nodes.add(s)
    for s in g.subjects(RDF.type, RDFS.Class):
        if isinstance(s, URIRef):
            class_nodes.add(s)

    for c in class_nodes:
        iri = str(c)
        cd = idx.classes.get(iri) or ClassDef(iri=iri)

        cd.labels = _collect_literals(g, c, RDFS.label)
        cd.comments = _collect_literals(g, c, RDFS.comment)

        cd.super_classes |= {str(o) for o in g.objects(c, RDFS.subClassOf) if isinstance(o, URIRef)}
        cd.equivalent_classes |= {str(o) for o in g.objects(c, OWL.equivalentClass) if isinstance(o, URIRef)}
        cd.disjoint_with |= {str(o) for o in g.objects(c, OWL.disjointWith) if isinstance(o, URIRef)}

        idx.classes[iri] = cd

    # -------- properties --------
    prop_nodes: Set[URIRef] = set()

    # collect any node that is declared a property via type
    property_type_iris = {
        RDF.Property,
        OWL.ObjectProperty,
        OWL.DatatypeProperty,
        OWL.AnnotationProperty,
        OWL.OntologyProperty,
        OWL.FunctionalProperty,
        OWL.InverseFunctionalProperty,
        OWL.SymmetricProperty,
        OWL.TransitiveProperty,
    }
    for t in property_type_iris:
        for s in g.subjects(RDF.type, t):
            if isinstance(s, URIRef):
                prop_nodes.add(s)

    # also include anything used as a predicate in the graph (schema files often omit explicit typing)
    for p in set(g.predicates()):
        if isinstance(p, URIRef):
            prop_nodes.add(p)

    for p in prop_nodes:
        iri = str(p)
        pd = idx.props.get(iri) or PropDef(iri=iri)

        pd.labels = _collect_literals(g, p, RDFS.label)
        pd.comments = _collect_literals(g, p, RDFS.comment)

        # all rdf:type signals (not just object/datatype)
        for t in g.objects(p, RDF.type):
            if isinstance(t, URIRef):
                pd.property_types.add(str(t))

        pd.domains |= {str(o) for o in g.objects(p, RDFS.domain) if isinstance(o, URIRef)}

        for o in g.objects(p, RDFS.range):
            if not isinstance(o, URIRef):
                continue
            o_str = str(o)
            if o_str.startswith(str(XSD)):
                pd.literal_datatypes.add(o_str) # xsd:string, xsd:integer, etc.
            else:
                pd.range_classes.add(o_str)     # dblp:Creator, dblp:Publication, etc.

        pd.super_properties |= {str(o) for o in g.objects(p, RDFS.subPropertyOf) if isinstance(o, URIRef)}
        pd.equivalent_properties |= {str(o) for o in g.objects(p, OWL.equivalentProperty) if isinstance(o, URIRef)}

        # owl:inverseOf is symmetric in practice, store both directions
        for o in g.objects(p, OWL.inverseOf):
            if isinstance(o, URIRef):
                pd.inverses.add(str(o))

        idx.props[iri] = pd

    # second pass: make inverseOf explicitly symmetric in the index
    for piri, pd in list(idx.props.items()):
        for inv in list(pd.inverses):
            inv_pd = idx.props.get(inv)
            if inv_pd is None:
                inv_pd = PropDef(iri=inv)
                idx.props[inv] = inv_pd
            inv_pd.inverses.add(piri)

    return idx


def _collect_literals(g: Graph, s: URIRef, p: URIRef) -> List[str]:
    out: List[str] = []
    for o in g.objects(s, p):
        if isinstance(o, Literal):
            val = str(o).strip()
            if val and val not in out:
                out.append(val)
    return out

In [ ]:
from typing import List, Dict, Optional, Iterable

# ----------------------------
# Compact context renderer
# ----------------------------

def _range_compact(p: PropDef, namespaces: Dict[str, str]) -> str:
    if p.literal_datatypes:
        dts = sorted(make_curie(dt, namespaces) for dt in p.literal_datatypes)
        return f"lit({','.join(dts)})"
    if p.range_classes:
        cs = sorted(make_curie(c, namespaces) for c in p.range_classes)
        return f"iri({'|'.join(cs)})"
    return "unk"


def _fmt_curie_list(iris: Iterable[str], namespaces: Dict[str, str], *, max_items: Optional[int] = None) -> str:
    items = [make_curie(i, namespaces) for i in iris]
    items = sorted(dict.fromkeys(items))  # stable dedupe
    if max_items is not None:
        items = items[:max_items]   # Can we somehow prioritize academic (scholarly) context first? for non-scholarly kgs
    return ", ".join(items)


def schema_to_context_string(
    idx: SchemaIndex,
    *,
    include_prefixes: bool = True,
    max_types: Optional[int] = None,
    max_preds_per_type: Optional[int] = None,
    include_notes: bool = True,
    include_predicates: bool = True,
    include_type_header: bool = True,
    include_pred_header: bool = True,
    include_class_relations: bool = True,
    include_superclasses: bool = True,
    include_equivalents: bool = True,
    max_rel_items: Optional[int] = None,
) -> str:
    lines: List[str] = []

    if include_prefixes:
        lines.append("PREFIXES")
        for pfx, ns in sorted(idx.namespaces.items()):
            lines.append(f"{pfx}: <{ns}>")
        lines.append("")

    lines.append("SCHEMA")
    lines.append("Legend: type | label | opt(note) ; pred | label | rng: lit(xsd:*) or iri(Type)")
    if include_class_relations:
        lines.append("Legend2: subClassOf: ..., equiv: ... (direct links only)")
    lines.append("")

    # domain -> predicates
    domain_to_preds: Dict[str, List[PropDef]] = {}
    if include_predicates:
        for p in idx.props.values():
            for d in p.domains:
                domain_to_preds.setdefault(d, []).append(p)

    # sorted types -> Maybe sort by frequency (prioritize classes with more properties or instances)
    types = sorted(idx.classes.values(), key=lambda c: make_curie(c.iri, idx.namespaces))
    if max_types is not None:
        types = types[:max_types]

    if include_type_header:
        lines.append("Types:")

    for c in types:
        c_id = make_curie(c.iri, idx.namespaces)
        c_label = c.labels[0] if c.labels else c_id.split(":")[-1]
        line = f"{c_id} | {c_label} | "
        if include_notes and c.comments:
            line += f"{';'.join(c.comments)}"
        lines.append(line)

        if include_class_relations:
            if include_superclasses:
                sups = getattr(c, "super_classes", set()) or set()
                if sups:
                    lines.append(f"  subClassOf: {_fmt_curie_list(sups, idx.namespaces, max_items=max_rel_items)}")

            if include_equivalents:
                eqs = getattr(c, "equivalent_classes", set()) or set()
                if eqs:
                    lines.append(f"  equiv: {_fmt_curie_list(eqs, idx.namespaces, max_items=max_rel_items)}")

        preds: List[PropDef] = []
        if include_predicates:
            preds = domain_to_preds.get(c.iri, [])
            preds.sort(key=lambda p: make_curie(p.iri, idx.namespaces))
            if max_preds_per_type is not None:
                preds = preds[:max_preds_per_type]

        if preds:
            if include_pred_header:
                lines.append("  Predicates:")
            for p in preds:
                p_id = make_curie(p.iri, idx.namespaces)
                p_label = p.labels[0] if p.labels else p_id.split(":")[-1]
                rng = _range_compact(p, idx.namespaces)
                lines.append(f"    {p_id} | {p_label} | rng:{rng}")

        lines.append("")

    return "\n".join(lines).rstrip()

In [ ]:
ctx = load_schema("./dblp_schema.rdf", base_iri="https://dblp.org/rdf/schema#")

# Build compact context string for the LLM
ctx_string = schema_to_context_string(
    ctx,
    include_prefixes=True,  
    include_notes=True,
    include_equivalents=False  
)

print(len(ctx_string))
print(ctx_string)

9666
PREFIXES
bf: <http://id.loc.gov/ontologies/bibframe/>
bibo: <http://purl.org/ontology/bibo/>
bibtex: <http://purl.org/net/nknouf/ns/bibtex#>
brick: <https://brickschema.org/schema/Brick#>
cito: <http://purl.org/spar/cito/>
csvw: <http://www.w3.org/ns/csvw#>
datacite: <http://purl.org/spar/datacite/>
dblp: <https://dblp.org/rdf/schema#>
dbo: <http://dbpedia.org/ontology/>
dc: <http://purl.org/dc/elements/1.1/>
dcam: <http://purl.org/dc/dcam/>
dcat: <http://www.w3.org/ns/dcat#>
dcmitype: <http://purl.org/dc/dcmitype/>
dcterms: <http://purl.org/dc/terms/>
doap: <http://usefulinc.com/ns/doap#>
foaf: <http://xmlns.com/foaf/0.1/>
geo: <http://www.opengis.net/ont/geosparql#>
litre: <http://purl.org/spar/literal/>
locid: <http://id.loc.gov/vocabulary/identifiers/>
locrel: <http://id.loc.gov/vocabulary/relators/>
odrl: <http://www.w3.org/ns/odrl/2/>
org: <http://www.w3.org/ns/org#>
owl: <http://www.w3.org/2002/07/owl#>
prof: <http://www.w3.org/ns/dx/prof/>
prov: <http://www.w3.org/ns/prov#

In [5]:
from typing import Dict

def _is_xsd(dt_iri: str) -> bool:
    return dt_iri.startswith("http://www.w3.org/2001/XMLSchema#")


def _xsd_local(dt_iri: str) -> str:
    # http://www.w3.org/2001/XMLSchema#string -> string
    if "#" in dt_iri:
        return dt_iri.rsplit("#", 1)[-1]
    return dt_iri.rsplit("/", 1)[-1]


def filter_schema(
    idx: SchemaIndex,
    *,
    keep_only_literal_or_labelable_object: bool = False,
    include_predicates: bool = True,
) -> SchemaIndex:
    """
    Simplified filter for SchemaIndex.

    If include_predicates is False:
      - returns SchemaIndex with same namespaces/classes, props = {}.

    If keep_only_literal_or_labelable_object is True:
      - keeps a predicate iff:
        (A) it has at least one literal datatype range (p.literal_datatypes), OR
        (B) it has at least one object range class AND at least one target class is "labelable",
            where "labelable" means: that class has at least one predicate with xsd:string range.

    Always drops predicates:
      - with no domains
      - with neither literal_datatypes nor range_classes

    Does not drop classes.
    Does not assume PropDef has is_object_property / is_datatype_property fields.
    """

    if not include_predicates:
        return SchemaIndex(
            namespaces=dict(idx.namespaces),
            classes=dict(idx.classes),
            props={},
        )

    # domain(class IRI) -> list[PropDef] (from original idx)
    domain_to_props: Dict[str, list] = {}
    for p in idx.props.values():
        for d in getattr(p, "domains", set()) or set():
            domain_to_props.setdefault(d, []).append(p)

    def class_is_labelable(class_iri: str) -> bool:
        for p2 in domain_to_props.get(class_iri, []):
            for dt in getattr(p2, "literal_datatypes", set()) or set():
                if _is_xsd(dt) and _xsd_local(dt).lower() == "string":
                    return True
        return False

    new_props: Dict[str, PropDef] = {}

    for p in idx.props.values():
        domains = getattr(p, "domains", set()) or set()
        if not domains:
            continue

        lit_dts = getattr(p, "literal_datatypes", set()) or set()
        rng_cls = getattr(p, "range_classes", set()) or set()

        has_lit = bool(lit_dts)
        has_obj = bool(rng_cls)

        if not has_lit and not has_obj:
            continue

        if keep_only_literal_or_labelable_object:
            if has_lit:
                keep = True
            else:
                keep = any(class_is_labelable(t) for t in rng_cls)
            if not keep:
                continue

        new_props[p.iri] = PropDef(
            iri=p.iri,
            labels=list(getattr(p, "labels", []) or []),
            comments=list(getattr(p, "comments", []) or []),
            domains=set(domains),
            literal_datatypes=set(lit_dts),
            range_classes=set(rng_cls),
        )

    return SchemaIndex(
        namespaces=dict(idx.namespaces),
        classes=dict(idx.classes),
        props=new_props,
    )

In [6]:
# Filter for mention extraction
ctx_me = filter_schema(
        ctx,
        keep_only_literal_or_labelable_object=True,
    )

print(schema_to_context_string(ctx_me, include_prefixes=False, include_notes=True , max_types=50, max_preds_per_type=50))

SCHEMA
Legend: type | label | opt(note) ; pred | label | rng: lit(xsd:*) or iri(Type)
Legend2: subClassOf: ..., equiv: ... (direct links only)

Types:
dblp:AmbiguousCreator | AmbiguousCreator | Not an actual creator, but an ambiguous proxy for an unknown number of unrelated actual creators of the same name. Associated publications do not have their true creators determined yet.
  subClassOf: dblp:Creator, wd:Q48522
  Predicates:
    dblp:possibleActualCreator | possibleActualCreator | rng:iri(dblp:Creator)

dblp:Article | Article | A journal article.
  subClassOf: bibo:AcademicArticle, dblp:Publication, schema:Article, wd:Q13442814
  equiv: dbo:Article

dblp:AuthorSignature | AuthorSignature | The information that links a publication to an author.
  subClassOf: dblp:Signature

dblp:Book | Book | A book or a thesis.
  subClassOf: dblp:Publication
  equiv: bibo:Book, dbo:Book, schema:Book, wd:Q571

dblp:Conference | Conference | A conference or workshop series.
  subClassOf: dblp:Stream


## Functions for the LLM to navigate the Schema

In [8]:
def list_classes(idx: SchemaIndex) -> List[Dict[str, str]]:
    """
    Returns all classes with iri and best-effort label.
    """
    out = []
    for c in idx.classes.values():
        label = c.labels[0] if c.labels else c.iri.split("/")[-1]
        out.append({
            "iri": c.iri,
            "label": label,
        })
    return sorted(out, key=lambda x: x["label"].lower())

def describe_class(idx: SchemaIndex, class_iri: str) -> Optional[Dict]:
    """
    Returns a structured description of a class.
    """
    c = idx.classes.get(class_iri)
    if c is None:
        return None

    return {
        "iri": c.iri,
        "labels": c.labels,
        "comments": c.comments,
        "super_classes": sorted(c.super_classes),
        "equivalent_classes": sorted(c.equivalent_classes),
        "disjoint_with": sorted(c.disjoint_with),
        "predicates": sorted(
            p.iri
            for p in idx.props.values()
            if class_iri in p.domains
        ),
    }

def list_properties(idx: SchemaIndex) -> List[Dict[str, str]]:
    """
    Returns all properties with iri, label, and compact range info.
    """
    out = []
    for p in idx.props.values():
        label = p.labels[0] if p.labels else p.iri.split("/")[-1]

        if p.literal_datatypes:
            rng = "literal"
        elif p.range_classes:
            rng = "iri"
        else:
            rng = "unknown"

        out.append({
            "iri": p.iri,
            "label": label,
            "range_kind": rng,
        })

    return sorted(out, key=lambda x: x["label"].lower())

def describe_property(idx: SchemaIndex, prop_iri: str) -> Optional[Dict]:
    """
    Returns a structured description of a property.
    """
    p = idx.props.get(prop_iri)
    if p is None:
        return None

    return {
        "iri": p.iri,
        "labels": p.labels,
        "comments": p.comments,
        "domains": sorted(p.domains),
        "range_classes": sorted(p.range_classes),
        "literal_datatypes": sorted(p.literal_datatypes),
        "super_properties": sorted(p.super_properties),
        "equivalent_properties": sorted(p.equivalent_properties),
        "inverse_properties": sorted(p.inverses),
        "property_types": sorted(p.property_types),
    }

In [13]:
# print out all classes
classes = list_classes(ctx)

print(f"Number of classes: {len(classes)}")
print("First 10 classes:")
for c in classes[:10]:
    print(f"- {c['label']} ({c['iri']})")

properties = list_properties(ctx)

print(f"Number of properties: {len(properties)}")
print("First 10 properties:")
for p in properties[:10]:
    print(f"- {p['label']} ({p['iri']}) - range: {p['range_kind']}")

# print out description of Book class
book_desc = describe_class(ctx, "https://dblp.org/rdf/schema#Book")
print("Description of Book class:")
print(book_desc)

# print out description of 'authoredBy' property
authored_by_desc = describe_property(ctx, "https://dblp.org/rdf/schema#authoredBy")
print("Description of 'authoredBy' property:")
print(authored_by_desc)

Number of classes: 24
First 10 classes:
- AmbiguousCreator (https://dblp.org/rdf/schema#AmbiguousCreator)
- Article (https://dblp.org/rdf/schema#Article)
- AuthorSignature (https://dblp.org/rdf/schema#AuthorSignature)
- Book (https://dblp.org/rdf/schema#Book)
- Conference (https://dblp.org/rdf/schema#Conference)
- Creator (https://dblp.org/rdf/schema#Creator)
- Data (https://dblp.org/rdf/schema#Data)
- Editorship (https://dblp.org/rdf/schema#Editorship)
- EditorSignature (https://dblp.org/rdf/schema#EditorSignature)
- Entity (https://dblp.org/rdf/schema#Entity)
Number of properties: 95
First 10 properties:
- 22-rdf-syntax-ns#type (http://www.w3.org/1999/02/22-rdf-syntax-ns#type) - range: unknown
- abstract (http://purl.org/dc/terms/abstract) - range: unknown
- affiliation (https://dblp.org/rdf/schema#affiliation) - range: literal
- archivedWebpage (https://dblp.org/rdf/schema#archivedWebpage) - range: iri
- authoredBy (https://dblp.org/rdf/schema#authoredBy) - range: iri
- authorOf (ht